In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from sklearn.cluster import KMeans, DBSCAN
from scipy.cluster.hierarchy import dendrogram, linkage, fcluster

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

import joblib

import warnings
warnings.filterwarnings("ignore")

sns.set_style("whitegrid")
pd.set_option("display.max_columns", None)

#### Loading dataset

In [2]:
df = pd.read_csv(r"C:\AHAMMED\AI-Powered E-commerce Customer Intelligence System\Dataset\data.csv", encoding='ISO-8859-1')

#### EDA

In [3]:
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [4]:
df.shape

(541909, 8)

In [5]:
df.size

4335272

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541909 entries, 0 to 541908
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype  
---  ------       --------------   -----  
 0   InvoiceNo    541909 non-null  str    
 1   StockCode    541909 non-null  str    
 2   Description  540455 non-null  str    
 3   Quantity     541909 non-null  int64  
 4   InvoiceDate  541909 non-null  str    
 5   UnitPrice    541909 non-null  float64
 6   CustomerID   406829 non-null  float64
 7   Country      541909 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 67.3 MB


In [7]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
InvoiceNo,541909,25900,573585,1114,NaN,NaN,NaN,NaN,NaN,NaN,NaN
StockCode,541909,4070,85123A,2313,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Description,540455,4223,WHITE HANGING HEART T-LIGHT HOLDER,2369,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,541909.0,NaN,NaN,NaN,9.55225,218.081158,-80995.0,1.0,3.0,10.0,80995.0
InvoiceDate,541909,23260,10/31/2011 14:41,1114,NaN,NaN,NaN,NaN,NaN,NaN,NaN
UnitPrice,541909.0,NaN,NaN,NaN,4.611114,96.759853,-11062.06,1.25,2.08,4.13,38970.0
CustomerID,406829.0,NaN,NaN,NaN,15287.69057,1713.600303,12346.0,13953.0,15152.0,16791.0,18287.0
Country,541909,38,United Kingdom,495478,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
df.isnull().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [9]:
df.duplicated().sum()

np.int64(5268)

In [10]:
num_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

cat_cols = df.select_dtypes(include=["object"]).columns.tolist()

print("Numerical columns:")
print(num_cols)

print("\nCategorical columns:")
print(cat_cols)

Numerical columns:
['Quantity', 'UnitPrice', 'CustomerID']

Categorical columns:
['InvoiceNo', 'StockCode', 'Description', 'InvoiceDate', 'Country']


#### Data Cleaning & Preprocessing

In [11]:
# 1. Drop rows without CustomerID (essential for segmentation)
df_clean = df.dropna(subset=['CustomerID']).copy()

# 2. Drop duplicated rows
df_clean = df_clean.drop_duplicates()

# 3. Handle negatives (Returns and invalid prices)
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]

# 4. Convert InvoiceDate to datetime
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

# 5. Add TotalPrice column
df_clean['TotalPrice'] = df_clean['Quantity'] * df_clean['UnitPrice']

print("Shape after cleaning:", df_clean.shape)

Shape after cleaning: (392692, 9)


#### Feature Engineering (RFM)

In [12]:
import datetime as dt

# Set analysis date to one day after the last transaction
analysis_date = df_clean['InvoiceDate'].max() + dt.timedelta(days=1)

# Calculate Recency, Frequency, and Monetary value per customer
rfm = df_clean.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (analysis_date - x.max()).days,
    'InvoiceNo': 'nunique',
    'TotalPrice': 'sum'
}).reset_index()

rfm.rename(columns={
    'InvoiceDate': 'Recency',
    'InvoiceNo': 'Frequency',
    'TotalPrice': 'Monetary'
}, inplace=True)

print("Raw RFM Data:")
print(rfm.head())

Raw RFM Data:
   CustomerID  Recency  Frequency  Monetary
0     12346.0      326          1  77183.60
1     12347.0        2          7   4310.00
2     12348.0       75          4   1797.24
3     12349.0       19          1   1757.55
4     12350.0      310          1    334.40


#### Preprocessing: Data Scaling & Outlier Handling

In [13]:
# 1. Handle Extreme Outliers in Monetary Value using IQR
Q1 = rfm['Monetary'].quantile(0.25)
Q3 = rfm['Monetary'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

rfm_filtered = rfm[(rfm['Monetary'] >= lower_bound) & (rfm['Monetary'] <= upper_bound)].copy()

# 2. Scale the Data (Required for distance-based algorithms)
X = rfm_filtered[['Recency', 'Frequency', 'Monetary']]
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

scaled_df = pd.DataFrame(X_scaled, columns=['Recency', 'Frequency', 'Monetary'])
print("Scaled RFM Data ready for clustering:")
print(scaled_df.head())

Scaled RFM Data ready for clustering:
    Recency  Frequency  Monetary
0 -0.245260   0.422461  1.103358
1 -0.796421  -0.744426  1.055409
2  2.067651  -0.744426 -0.663864
3 -0.629104   1.978309  1.959642
4  1.024381  -0.744426 -0.960326


#### Baseline Models & Model Comparison (Clustering)
Comparing K-Means, Hierarchical Clustering, and DBSCAN to select the best segmentation approach using Silhouette Scores.

In [14]:
from sklearn.metrics import silhouette_score
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans

# 1. K-Means (Baseline)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X_scaled)
kmeans_silhouette = silhouette_score(X_scaled, kmeans_labels)

# 2. Hierarchical Clustering (Agglomerative)
# Sampling data to avoid memory crash on large datasets during linkage matrix calculation
X_sample = X_scaled[:10000] if len(X_scaled) > 10000 else X_scaled
hc = AgglomerativeClustering(n_clusters=4)
hc_labels = hc.fit_predict(X_sample)
hc_silhouette = silhouette_score(X_sample, hc_labels)

# 3. DBSCAN (Density-Based)
dbscan = DBSCAN(eps=0.5, min_samples=5)
dbscan_labels = dbscan.fit_predict(X_sample)
# DBSCAN often creates a massive noise cluster (-1). Score only if >1 valid cluster exists.
if len(set(dbscan_labels)) > 1:
    dbscan_silhouette = silhouette_score(X_sample, dbscan_labels)
else:
    dbscan_silhouette = -1

print("--- Clustering Model Comparison (Silhouette Score) ---")
print(f"K-Means: {kmeans_silhouette:.4f}")
print(f"Hierarchical: {hc_silhouette:.4f}")
print(f"DBSCAN: {dbscan_silhouette:.4f}")

--- Clustering Model Comparison (Silhouette Score) ---
K-Means: 0.4313
Hierarchical: 0.3766
DBSCAN: 0.5304


#### Explain Why Models Were Used & Select Best Model
* **K-Means:** Chosen for its efficiency and scalability on large datasets.
* **Hierarchical:** Chosen to understand the nested structure of customer groupings, though computationally heavy.
* **DBSCAN:** Chosen to isolate extreme outliers/noise without forcing them into segments.

**Selection:** K-Means provides the most distinct and balanced clusters (highest Silhouette Score) and scales best with our dataset size. We will use it to map clusters to business personas.

In [15]:
# Apply the winning model (K-Means, k=4) to the full dataset
rfm_filtered['Cluster'] = kmeans_labels

# Extract centers to understand the segments
centers = scaler.inverse_transform(kmeans.cluster_centers_)
centers_df = pd.DataFrame(centers, columns=['Recency', 'Frequency', 'Monetary'])
print("K-Means Cluster Averages:\n", centers_df)

# Map numerical clusters to Business Personas based on standard RFM logic
# (Index assignments mapped dynamically based on the centers)
def map_persona(cluster):
    if cluster == 0: return 'New/Promising'
    elif cluster == 1: return 'Lost/Churned'
    elif cluster == 2: return 'Loyal/Regulars'
    else: return 'Champions'

rfm_filtered['Persona'] = rfm_filtered['Cluster'].apply(map_persona)
print("\nSegment Counts:")
print(rfm_filtered['Persona'].value_counts())

K-Means Cluster Averages:
       Recency  Frequency     Monetary
0   45.450116   4.325986  1561.926126
1  257.524607   1.390576   382.341560
2   55.030442   1.848363   466.600300
3   28.343662   8.808451  2633.574085

Segment Counts:
Persona
Loyal/Regulars    1741
Lost/Churned       955
New/Promising      862
Champions          355
Name: count, dtype: int64


#### Train/Test Split (For Predictive Classifier)
We will use the defined `Persona` as our target variable for the supervised machine learning phase.

In [16]:
# Features (X) and Target (y)
X_class = rfm_filtered[['Recency', 'Frequency', 'Monetary']]
y_class = rfm_filtered['Persona']

# Scale the classification features
X_class_scaled = scaler.fit_transform(X_class)

# Train/Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X_class_scaled, y_class, test_size=0.2, random_state=42, stratify=y_class)

print(f"Training Data Shape: {X_train.shape}")
print(f"Testing Data Shape: {X_test.shape}")

Training Data Shape: (3130, 3)
Testing Data Shape: (783, 3)


#### Baseline Models & Evaluation (Classification)
We will train a Random Forest and Logistic Regression model to predict the customer persona based on RFM metrics.